# Text editor tool

Claude's **built-in** text editor tool. Anthropic defines the schema (`type="text_editor_20250728"`) and the command API — `view`, `str_replace`, `create`, `insert`, `undo_edit` with their parameters.
Here we wrap a `TextEditorTool` class that executes them against a sandboxed base directory.

## Safety

- `_validate_path` resolves paths against `base_dir` and refuses anything that escapes it.
- Every destructive op (`str_replace`, `insert`, `undo_edit`) backs the file up first into `.backups/` with the mtime as suffix; `undo_edit` restores the most recent backup.
- `str_replace` requires exactly one match: zero or multiple matches raise, so Claude is forced to supply enough context.

Model: `claude-sonnet-4-6`.
## Getting schemas for Anthropic tools

For user-implemented tools you write both the schema and the handler. For Anthropic's built-in tools you don't write the schema:
Anthropic publishes versioned schemas that you reference by `type` (e.g. `text_editor_20250728`, `web_search_20250305`), plus optional config fields (`max_uses`, `allowed_domains`, ...).
You pass the minimal stub and Anthropic expands the full schema internally.

Examples:
[Text editor](https://platform.claude.com/docs/en/agents-and-tools/tool-use/text-editor-tool),
[Web search](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool),
[Code execution](https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool),
[Computer use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/computer-use-tool),
[Bash](https://platform.claude.com/docs/en/agents-and-tools/tool-use/bash-tool),
[Memory](https://platform.claude.com/docs/en/agents-and-tools/tool-use/memory-tool).


In [33]:
import json
import os
import shutil

from chat_tools import add_assistant_message, add_user_message, chat, text_from_message
from models import MODEL_SONNET

In [34]:
class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(f"Access denied: Path '{file_path}' is outside the allowed directory")
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}")
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")
        latest_backup = sorted(backups, reverse=True)[0]
        shutil.copy2(os.path.join(self.backup_dir, latest_backup), file_path)
        return f"Successfully restored {file_path} from backup"

    def view(self, file_path: str, view_range: list[int] | None = None) -> str:
        abs_path = self._validate_path(file_path)

        if os.path.isdir(abs_path):
            return "\n".join(os.listdir(abs_path))

        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")

        with open(abs_path, encoding="utf-8") as f:
            content = f.read()

        lines = content.split("\n")
        if view_range:
            start, end = view_range
            if end == -1:
                end = len(lines)
            selected = lines[start - 1 : end]
            return "\n".join(f"{i}: {line}" for i, line in enumerate(selected, start))
        return "\n".join(f"{i}: {line}" for i, line in enumerate(lines, 1))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        abs_path = self._validate_path(file_path)

        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")

        with open(abs_path, encoding="utf-8") as f:
            content = f.read()

        match_count = content.count(old_str)
        if match_count == 0:
            raise ValueError("No match found for replacement. Please check your text and try again.")
        if match_count > 1:
            raise ValueError(
                f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
            )

        self._backup_file(abs_path)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(content.replace(old_str, new_str))
        return "Successfully replaced text at exactly one location."

    def create(self, file_path: str, file_text: str) -> str:
        abs_path = self._validate_path(file_path)

        if os.path.exists(abs_path):
            raise FileExistsError("File already exists. Use str_replace to modify it.")

        os.makedirs(os.path.dirname(abs_path), exist_ok=True)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(file_text)
        return f"Successfully created {file_path}"

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        abs_path = self._validate_path(file_path)

        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")

        self._backup_file(abs_path)
        with open(abs_path, encoding="utf-8") as f:
            lines = f.readlines()

        if lines and not lines[-1].endswith("\n"):
            new_str = "\n" + new_str

        if insert_line == 0:
            lines.insert(0, new_str + "\n")
        elif 0 < insert_line <= len(lines):
            lines.insert(insert_line, new_str + "\n")
        else:
            raise IndexError(f"Line number {insert_line} is out of range. File has {len(lines)} lines.")

        with open(abs_path, "w", encoding="utf-8") as f:
            f.writelines(lines)
        return f"Successfully inserted text after line {insert_line}"

    def undo_edit(self, file_path: str) -> str:
        abs_path = self._validate_path(file_path)

        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")

        return self._restore_backup(abs_path)

In [35]:
text_editor_tool = TextEditorTool()


def run_tool(tool_name, tool_input):
    if tool_name != "str_replace_based_edit_tool":
        raise ValueError(f"Unknown tool: {tool_name}")

    command = tool_input["command"]
    if command == "view":
        return text_editor_tool.view(tool_input["path"], tool_input.get("view_range"))
    if command == "str_replace":
        return text_editor_tool.str_replace(tool_input["path"], tool_input["old_str"], tool_input["new_str"])
    if command == "create":
        return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
    if command == "insert":
        return text_editor_tool.insert(tool_input["path"], tool_input["insert_line"], tool_input["new_str"])
    if command == "undo_edit":
        return text_editor_tool.undo_edit(tool_input["path"])
    raise ValueError(f"Unknown text editor command: {command}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [36]:
# Minimal schema stub: Anthropic expands it internally. The type string is model-version specific.
text_editor_schema = {
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",
}

In [37]:
def run_conversation(messages):
    while True:
        response = chat(messages, model=MODEL_SONNET, tools=[text_editor_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            print("---\n")
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [38]:
messages = []
add_user_message(
    messages,
    " ".join(
        [
            "Create a file called hello.txt in './' with the content 'Hello, world!'",
            "and then show me its contents.",
        ]
    ),
)

run_conversation(messages)

Sure! Let me do both steps at the same time — create the file and view it!
Here's a summary of what was done:

1. **Created** `./hello.txt` with the content `Hello, world!` ✅
2. **Viewed** the file contents ✅

The file has a single line:
```
Hello, world!
```
---



[{'role': 'user',
  'content': "Create a file called hello.txt in './' with the content 'Hello, world!' and then show me its contents."},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Sure! Let me do both steps at the same time — create the file and view it!', type='text'),
   ToolUseBlock(id='toolu_01JZzVszCCxuCCt1MMdk8ucN', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': './hello.txt', 'file_text': 'Hello, world!'}, name='str_replace_based_edit_tool', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_01KKnm3TmPJD6bPGqoZ6U8S5', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './hello.txt'}, name='str_replace_based_edit_tool', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01JZzVszCCxuCCt1MMdk8ucN',
    'content': '"Successfully created ./hello.txt"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01KKn